# 第 09 章练习

## 9.7 章节实践

先完成基础题和两个实验的对照设计，再从 GMRES、MuduoXinyu 中任选一个做运行分析。只运行所选部分的目录设置和代码单元格。

下面两组专项练习任选一组。本 Notebook 始终从第 09 章目录运行，使用原实验已准备好的源码和模型。


<a id="s-09-02_chapter_test"></a>


### 一、基础理解

1. GMRES 的残差是什么？restart 改变哪些计算和存储开销？
2. GMRES 的外层控制与 SpMV/Dot/AXPY 分别在哪里运行？
3. Attention 为什么需要 KV cache？融合算子主要减少什么开销？
4. Muduo A/B 除了算子组织，还改变了什么？
5. 如何区分功能通过、数据可比较和测到性能收益？
6. 为什么生成 token 一致，不要求每个中间浮点数都完全相同？
7. 多次测量的均值和波动分别帮助判断什么？

[综合题参考答案](answer/09.07_chapter_test_answer.md)；[MuduoXinyu 参考答案](02_muduoxinyu_optimization/answer/09.07_muduo_answer.md)


### 二、说明比较方法

说明怎样建立基线、保持比较条件一致、检查正确性并判断性能变化。再列出 MuduoXinyu 的两项功能检查，例如 token 一致性、是否发生 fallback、FlashAttention 调用次数。

### 三、设计对照

- GMRES：单卡比较 restart=4 与 30，固定输入、MGS、容差与最大迭代数。记录实际迭代数、残差和解误差，再比较总时间。
- Muduo：固定模型、提示词与生成长度，每条路径 1 次预热、3 次统计。说明 token 一致性、工作量和路径检查，并记录吞吐量均值与波动。

写清修改项、保持不变的条件、检查方法和要保存的结果。

### 四、运行并分析

任选一项完成实验：

- GMRES：采集单卡 Profile，找到主要耗时，比较 restart=4 与 30，记录正确性和时间。
- MuduoXinyu：运行两种 Attention 实现，完成各 1 次预热和 3 次统计，记录功能检查、数据有效性和吞吐量变化。

提交命令、日志摘要、结果表和分析。说明本次是否加速，以及测量使用的输入、设备和计时范围。

<a id="s-01-06_chapter_test"></a>


下面运行 GMRES 的重启长度对照。命令已包含子实验路径。


## GMRES：比较 restart（任选）

下面使用 U2，保持其他求解参数相同，比较 restart=4 与 30。分析时使用同一矩阵采集的 Profile。

### 检查设备和工具

下面显示设备信息和 CANN 安装位置，并检查本节要用的命令是否能找到。

In [ ]:
%%bash
set -e
echo "CANN 目录：$ASCEND_HOME_PATH"
cmake --version
npu-smi info
command -v msprof


### 实践任务

单卡比较 `restart=4` 与 `restart=30`：矩阵、容差、最大迭代数和重复次数相同，只改变重启长度。
记录真实残差、解误差、实际迭代数、total 和 Dot/AXPY 时间，解释存储与正交化开销的取舍。
多卡时可另外比较 rows/nnz 分区。

### 运行并记录结果

执行下面的命令，保存本次输出。

In [ ]:
%%bash
set -euo pipefail
cd 01_distributed_gmres/src/dis_gmres
cmake -S . -B build -DCMAKE_BUILD_TYPE=Release -DDIS_GMRES_FORCE_STUB=OFF
cmake --build build -j4
mkdir -p results/restart_compare
# 学生修改：要比较的 restart；本题先使用 4 和 30。
for restart in 4 30; do
  ./build/bin/dis_gmres --matrix U2 --rank 0 --world-size 1 --device 0 \
    --warmup 0 --repeat 3 --restart "$restart" --max-iterations 300 --tolerance 1e-6 \
    --orthogonalization mgs --result-csv "results/restart_compare/r${restart}.csv" \
    2>&1 | tee "results/restart_compare/r${restart}.log"
done


### GMRES 分析

解释 SpMV、Dot、AXPY、Norm 在 Arnoldi 中的作用。结合本次残差、解误差、实际迭代数和 total，分析改变 restart 后的差异。API 与 task 表分别统计，注明来源文件和可能重叠的时间。

[GMRES 参考答案](01_distributed_gmres/answer/09.07_gmres_answer.md)
若本次 restart=4 比 30 迭代更少，请结合实验 Notebook 的残差检查流程分析。汇总日志能说明最终是否收敛；若要区分舍入误差与检查时机的影响，还需逐步残差和正交性记录。


两次单卡输出的 `Speedup=1`，是各配置相对自身单卡基线的比值。本题比较配置应使用 `T(restart=30)/T(restart=4)`；现成结果约为 `10826/3733=2.90`，同时还要比较迭代数与真实残差。

阶段时间是一次 solve 内多次调用的合计，不是单次 SpMV。与第 06 章比较时，矩阵规模、调用次数和计时范围都要对应。


<a id="s-02-03_chapter_test"></a>


下面运行 MuduoXinyu 的 A/B 实验。路径相对于第 09 章目录。


## MuduoXinyu：运行与分析 A/B（任选）

保持模型、提示词和生成长度一致，分别检查功能、数据有效性和吞吐量变化。

### 设置实验路径

使用前面编译好的源码、模型和 tokenizer；放在其他位置时修改路径。


In [ ]:
# 文件已放在其他位置时修改路径。
%env MUDUO_ROOT=02_muduoxinyu_optimization/muduoxinyu-demo
%env MODEL=02_muduoxinyu_optimization/assets/stories110M.bin
%env TOKENIZER=02_muduoxinyu_optimization/assets/tokenizer.bin
%env OUTPUT_DIR=02_muduoxinyu_optimization/results/attention_practice


### 三、运行 A/B

直接运行下面的命令，保存本次输出。模型、提示词、生成长度和统计次数由同一个脚本设置。


| 条件 | 脚本默认值 |
|---|---|
| 提示词 | hello |
| 短功能检查 | 每条路径 12 步 |
| 性能测试 | 每条路径 120 步；四个相同提示词样本 |
| 解码 | temperature=0，topP=1 |
| 统计 | 第一个样本预热，后三个求均值和 CV |

CV 是标准差除以均值的百分比，表示这几个样本的波动。改变生成长度时，修改 `run_ab_benchmark.sh` 开头的 `PERF_STEPS`，再重跑两种实现。


In [ ]:
%%bash
bash 02_muduoxinyu_optimization/src/run_ab_benchmark.sh \
  --muduo-root "$MUDUO_ROOT" \
  --model "$MODEL" --tokenizer "$TOKENIZER" \
  --output-dir "$OUTPUT_DIR"


### 四、困难编程题：复合变量解释

结合调用次数、mean、CV、dtype/Cast、布局与 launch 开销，分别回答：功能是否通过、数据是否有效、性能是否受益。然后设计一个后续单变量实验，例如只固定 dtype 或只改变运行顺序；根据本次数据说明快慢，并保留原始日志。

In [ ]:
import json
import os
from pathlib import Path

result = json.loads((Path(os.environ["OUTPUT_DIR"]) / "result.json").read_text())
print("功能检查通过：", result["verdict"]["functional_pass"])
print("性能数据有效：", result["verdict"]["performance_valid"])
print("本批 B 均值更高：", result["verdict"]["performance_beneficial"])
for path in ("path_a", "path_b"):
    perf = result["perf"][path]
    print(path, f"{perf['mean_tokens_per_s']:.2f} tokens/s，波动 {perf['cv_percent']:.2f}%")
print(f"B 相对 A 的吞吐量变化：{result['perf']['delta_percent_b_vs_a']:.2f}%")

print("短检查调用次数：", result["smoke"]["flash_stats"])
print("短检查结果：", result["smoke"]["checks"])


### 完成标准

解释功能检查和吞吐量变化，说明本实验比较了 FP32 多算子与 FP16 FlashAttention 两套实现。没有加速时也按本次结果分析原因。


完成本节练习后查看：[参考答案](02_muduoxinyu_optimization/answer/09.07_muduo_answer.md)